# Tier 3 Solutions Set 1: NGS Pipelines

Solutions for Assignment Set 1: NGS Pipelines.

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
import numpy as np
import math
from collections import defaultdict
from scipy import stats

---

## Problem 1: FASTQ Quality Parser (1 star)

In [ ]:
def parse_fastq(fastq_string: str) -> list[dict]:
    """
    Parse a FASTQ-format string and return a list of read records.

    Args:
        fastq_string: Multi-record FASTQ string (4 lines per record).

    Returns:
        List of dicts with keys: 'id', 'sequence', 'quality'.
    """
    records = []
    lines = [line for line in fastq_string.splitlines() if line.strip()]

    for i in range(0, len(lines), 4):
        read_id  = lines[i].lstrip('@')
        sequence = lines[i + 1]
        # lines[i + 2] is the '+' separator — ignored
        quality  = [ord(c) - 33 for c in lines[i + 3]]
        records.append({'id': read_id, 'sequence': sequence, 'quality': quality})

    return records


# Test
sample_fastq = """@read1
ACGTACGTACGT
+
IIIIIIIIIIII
@read2
TTTTAAAACCCC
+
AAAABBBBCCCC
@read3
GCGCGCGCGCGC
+
??????????@@
"""

records = parse_fastq(sample_fastq)
for r in records:
    print(f"ID: {r['id']}, Seq: {r['sequence']}, Qual: {r['quality']}")

---

## Problem 2: Quality Trimming (1 star)

In [ ]:
def sliding_window_trim(
    sequence: str,
    quality: list[int],
    window_size: int = 5,
    min_quality: int = 20,
) -> tuple[str, int]:
    """
    Trim a read from the 3' end using a sliding window quality filter.

    Scans from the 3' end; removes bases until the window average >= min_quality.

    Args:
        sequence: DNA sequence string.
        quality: List of Phred quality scores, same length as sequence.
        window_size: Size of the sliding window.
        min_quality: Minimum average quality to retain a window.

    Returns:
        Tuple of (trimmed_sequence, trimmed_length).
    """
    trim_pos = len(sequence)  # start assuming no trimming needed

    # Walk from 3' end inward; find the rightmost position where the
    # window starting there meets the quality threshold.
    for i in range(len(sequence) - window_size, -1, -1):
        window_avg = sum(quality[i:i + window_size]) / window_size
        if window_avg >= min_quality:
            trim_pos = i + window_size
            break
    else:
        # No window passed — trim everything
        trim_pos = 0

    trimmed = sequence[:trim_pos]
    return trimmed, len(trimmed)


# Test
seq = 'ACGTACGTACGT'
quals = [30, 30, 30, 30, 30, 30, 30, 15, 10, 8, 5, 3]
trimmed, length = sliding_window_trim(seq, quals)
print(f"Trimmed sequence: {trimmed}")
print(f"Trimmed length:   {length}")

---

## Problem 3: VCF Parser (2 stars)

In [ ]:
def parse_vcf(vcf_string: str) -> list[dict]:
    """
    Parse a VCF-format string and return a list of variant records.

    Args:
        vcf_string: Full VCF content including header lines.

    Returns:
        List of dicts with keys: 'chrom', 'pos', 'ref', 'alt', 'qual', 'filter'.
    """
    variants = []
    for line in vcf_string.splitlines():
        if not line.strip() or line.startswith('#'):
            continue
        fields = line.split('\t')
        qual_raw = fields[5]
        variants.append({
            'chrom':  fields[0],
            'pos':    int(fields[1]),
            'ref':    fields[3],
            'alt':    fields[4],
            'qual':   None if qual_raw == '.' else float(qual_raw),
            'filter': fields[6],
        })
    return variants


# Test
sample_vcf = """##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr1\t12345\t.\tA\tT\t50.0\tPASS\t.
chr1\t67890\trs123\tGG\tA\t30.5\tPASS\t.
chr2\t11111\t.\tC\tG\t.\tLowQual\t.
chr3\t99999\t.\tT\tA,C\t75.2\tPASS\t.
"""

variants = parse_vcf(sample_vcf)
for v in variants:
    print(v)

---

## Problem 4: Variant Annotation (2 stars)

In [ ]:
def annotate_variants(
    variants: list[dict],
    genes: list[dict],
) -> list[dict]:
    """
    Annotate variants with the gene they fall in.

    Args:
        variants: List of dicts with 'chrom' and 'pos' keys.
        genes: List of dicts with 'gene', 'chrom', 'start', 'end' keys.
               Coordinates are 1-based, inclusive.

    Returns:
        List of variant dicts with an added 'gene' key.
    """
    annotated = []
    for variant in variants:
        matched_gene = 'intergenic'
        for gene in genes:
            if (
                gene['chrom'] == variant['chrom']
                and gene['start'] <= variant['pos'] <= gene['end']
            ):
                matched_gene = gene['gene']
                break
        annotated.append({**variant, 'gene': matched_gene})
    return annotated


# Test
variants = [
    {'chrom': 'chr1', 'pos': 150},
    {'chrom': 'chr1', 'pos': 500},
    {'chrom': 'chr2', 'pos': 300},
    {'chrom': 'chr1', 'pos': 50},
]
genes = [
    {'gene': 'GENE_A', 'chrom': 'chr1', 'start': 100, 'end': 200},
    {'gene': 'GENE_B', 'chrom': 'chr1', 'start': 400, 'end': 600},
    {'gene': 'GENE_C', 'chrom': 'chr2', 'start': 250, 'end': 350},
]

annotated = annotate_variants(variants, genes)
for v in annotated:
    print(v)

---

## Problem 5: Read Depth Calculator (2 stars)

In [ ]:
def calculate_coverage(
    reads: list[tuple[int, int]],
    ref_length: int,
) -> dict:
    """
    Calculate per-position read depth and coverage statistics.

    Args:
        reads: List of (start, end) tuples. 0-based, end-exclusive.
        ref_length: Total length of the reference sequence.

    Returns:
        Dict with keys:
            'depth': list of per-position depth values
            'mean_depth': float
            'zero_coverage_positions': list of positions with depth == 0
            'fraction_10x': fraction of positions with depth >= 10
    """
    # Use a difference array for O(n + r) time instead of O(n * r)
    diff = [0] * (ref_length + 1)
    for start, end in reads:
        diff[start] += 1
        if end < len(diff):
            diff[end] -= 1

    depth = []
    current = 0
    for i in range(ref_length):
        current += diff[i]
        depth.append(current)

    zero_positions = [i for i, d in enumerate(depth) if d == 0]
    fraction_10x   = sum(1 for d in depth if d >= 10) / ref_length

    return {
        'depth': depth,
        'mean_depth': sum(depth) / ref_length,
        'zero_coverage_positions': zero_positions,
        'fraction_10x': fraction_10x,
    }


# Test
reads = [(0, 50), (30, 80), (75, 120)]
ref_length = 150

result = calculate_coverage(reads, ref_length)
print(f"Mean depth:              {result['mean_depth']:.2f}")
print(f"Zero-coverage positions: {result['zero_coverage_positions']}")
print(f"Fraction at >=10x:       {result['fraction_10x']:.3f}")

---

## Problem 6: Simple Differential Expression (3 stars)

In [ ]:
def _bh_correction(pvalues: list[float]) -> list[float]:
    """Benjamini-Hochberg FDR correction (internal helper)."""
    n = len(pvalues)
    order = sorted(range(n), key=lambda i: pvalues[i])
    adj = [0.0] * n

    for rank, idx in enumerate(order, start=1):
        adj[idx] = min(pvalues[idx] * n / rank, 1.0)

    # Enforce monotonicity: step backwards through sorted order
    for i in range(len(order) - 2, -1, -1):
        adj[order[i]] = min(adj[order[i]], adj[order[i + 1]])

    return adj


def differential_expression(
    matrix_a: np.ndarray,
    matrix_b: np.ndarray,
    pseudocount: float = 1.0,
    alpha: float = 0.05,
    min_log2fc: float = 1.0,
) -> list[dict]:
    """
    Identify differentially expressed genes between two conditions.

    Args:
        matrix_a: Gene expression matrix for condition A, shape (n_genes, n_samples).
        matrix_b: Gene expression matrix for condition B, shape (n_genes, n_samples).
        pseudocount: Added to means before log2 to avoid log(0).
        alpha: FDR significance threshold.
        min_log2fc: Minimum absolute log2 fold change threshold.

    Returns:
        List of dicts for significant genes, each with:
        'gene_idx', 'log2fc', 'pval', 'adj_pval'.
    """
    n_genes = matrix_a.shape[0]
    log2fc_values = []
    pvalues = []

    for g in range(n_genes):
        a_vals = matrix_a[g]
        b_vals = matrix_b[g]
        mean_a = np.mean(a_vals) + pseudocount
        mean_b = np.mean(b_vals) + pseudocount
        log2fc = np.log2(mean_b / mean_a)
        _, pval = stats.ttest_ind(a_vals, b_vals)
        log2fc_values.append(log2fc)
        pvalues.append(float(pval))

    adj_pvalues = _bh_correction(pvalues)

    significant = []
    for g in range(n_genes):
        if adj_pvalues[g] < alpha and abs(log2fc_values[g]) > min_log2fc:
            significant.append({
                'gene_idx': g,
                'log2fc':   round(log2fc_values[g], 4),
                'pval':     round(pvalues[g], 6),
                'adj_pval': round(adj_pvalues[g], 6),
            })

    significant.sort(key=lambda x: x['adj_pval'])
    return significant


# Test
np.random.seed(42)
n_genes, n_samples = 100, 6

cond_a = np.random.exponential(scale=10, size=(n_genes, n_samples))
cond_b = cond_a.copy()
cond_b[:10] *= 8
cond_b[10:15] *= 0.1

sig_genes = differential_expression(cond_a, cond_b)
print(f"Significant genes: {len(sig_genes)}")
for g in sig_genes[:5]:
    print(f"  Gene {g['gene_idx']:3d}: log2FC={g['log2fc']:+.2f}, adj_p={g['adj_pval']:.4f}")

---

## Problem 7: 16S Diversity Metrics (3 stars)

In [ ]:
def shannon_diversity(sample: np.ndarray) -> float:
    """
    Calculate Shannon diversity index for a single sample.

    Args:
        sample: 1D array of OTU abundances for one sample.

    Returns:
        Shannon H value.
    """
    total = sample.sum()
    if total == 0:
        return 0.0
    proportions = sample[sample > 0] / total
    return float(-np.sum(proportions * np.log(proportions)))


def simpson_diversity(sample: np.ndarray) -> float:
    """
    Calculate Simpson's diversity index (1 - D) for a single sample.

    Args:
        sample: 1D array of OTU abundances for one sample.

    Returns:
        Simpson 1-D value (higher = more diverse).
    """
    total = sample.sum()
    if total == 0:
        return 0.0
    proportions = sample / total
    return float(1.0 - np.sum(proportions ** 2))


def bray_curtis_matrix(otu_table: np.ndarray) -> np.ndarray:
    """
    Compute pairwise Bray-Curtis dissimilarity matrix.

    Args:
        otu_table: 2D array of shape (n_otus, n_samples).

    Returns:
        Symmetric dissimilarity matrix of shape (n_samples, n_samples).
    """
    n_samples = otu_table.shape[1]
    bc = np.zeros((n_samples, n_samples))

    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            x = otu_table[:, i]
            y = otu_table[:, j]
            denom = x.sum() + y.sum()
            if denom == 0:
                bc[i, j] = bc[j, i] = 0.0
            else:
                bc[i, j] = bc[j, i] = np.sum(np.abs(x - y)) / denom

    return bc


# Test
np.random.seed(7)
otu_table = np.random.randint(0, 100, size=(20, 5)).astype(float)
otu_table[:, 2] = np.random.randint(0, 5, size=20)

print("Shannon diversity per sample:")
for i in range(otu_table.shape[1]):
    print(f"  Sample {i}: {shannon_diversity(otu_table[:, i]):.3f}")

print("\nSimpson diversity per sample:")
for i in range(otu_table.shape[1]):
    print(f"  Sample {i}: {simpson_diversity(otu_table[:, i]):.3f}")

bc = bray_curtis_matrix(otu_table)
print("\nBray-Curtis dissimilarity matrix:")
print(np.round(bc, 3))

---

## Summary

Key implementation notes:

1. **FASTQ Parsing**: Phred+33 decoding is `ord(char) - 33`
2. **Quality Trimming**: Scan inward from 3' end; stop at the first window that meets the threshold
3. **VCF Parsing**: Handle `.` as `None` for missing QUAL; `pos` is an `int`
4. **Variant Annotation**: Check chromosome match before coordinate overlap
5. **Coverage**: Difference array gives O(n + r) depth calculation
6. **DE Analysis**: BH requires monotonicity enforcement in reverse rank order
7. **Diversity**: Shannon uses natural log; Bray-Curtis is symmetric so only compute upper triangle